# CP-1: YOLOv8l comparison run

Run all cells in order in **Google Colab with a GPU**, using the same
`/content/drive/MyDrive/CP-1 Dataset` folder as your YOLO11 run.
This notebook uses pretrained `yolov8l.pt`; it does not train the detector.
Model reference: https://docs.ultralytics.com/models/yolov8/

The exact 80 source clips, future labels and 48/32 date split from your supplied
YOLO11 v2 export are embedded below. No additional CSV upload is needed.
Keep the counting lines, thresholds, inference sizes, tracker and XGBoost
settings unchanged for the comparison. Large variants are not parameter-matched.
Package versions/hardware may differ from the historical run; this is not a
controlled runtime benchmark.

All generated files use YOLOv8-specific locations; YOLO11 results are untouched.
Completed YOLOv8 clips can resume after a disconnect. Run cells from the top
after reconnecting. Do not change settings while reusing a checkpoint.
Cell 6 requires all 80 clips before scoring. The final cell creates
`Journal_Results_Export_yolov8_v2.zip` in your dataset folder.

Counts and confidence scores are not detection accuracy. mAP, MOTA and count
RMSE still require manual ground truth. The XGBoost evaluation measures
next-segment congestion classification, not detector accuracy.


In [ ]:
#-- cell 1
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")


In [ ]:
#-- cell 2
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Install required packages ---
# YOLOv8 requires a modern Ultralytics package. The weights download automatically on first use.
!pip install -U "ultralytics>=8.3.0" "xgboost>=2.0.0" "scikit-learn>=1.4" -q

# --- Verify GPU ---
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("?? WARNING: No GPU detected! Go to Runtime ? Change runtime type ? T4 GPU")

# --- Import all libraries ---
import os
import cv2
import numpy as np
import pandas as pd
from glob import glob
from pathlib import Path
from ultralytics import YOLO
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("\n? All packages installed and imported successfully!")

In [ ]:
# --- Project configuration (Google Colab) ---
DRIVE_BASE = "/content/drive/MyDrive/CP-1 Dataset"
YOLO_MODEL_NAME = "yolov8l.pt"  # Pretrained COCO detector; not trained on this dataset.
RESUME_FEATURE_EXTRACTION = True  # Completed clips only; partial tracker state is not recoverable.
FEATURES_CSV_PATH = os.path.join(DRIVE_BASE, "extracted_features_yolov8_v2.csv")
ANNOTATION_DIR = os.path.join(DRIVE_BASE, "annotated_samples_yolov8_v2")
RUN_LOG_DIR = os.path.join(DRIVE_BASE, "crossing_logs_yolov8_v2")
JOURNAL_EXPORT_DIR = os.path.join(DRIVE_BASE, "Journal_Results_Export_yolov8_v2")
TRAIN_CSV_PATH = os.path.join(DRIVE_BASE, "CSV", "Train.csv")
SAMPLE_SUBMISSION_PATH = os.path.join(DRIVE_BASE, "CSV", "SampleSubmission.csv")
TRACKER_CONFIG_PATH = "custom_bytetrack_yolov8.yaml"

# Source clips are sampled by their *next* labeled time segment. These defaults
# select up to 3 training and 2 held-out clips per camera and future class.
TRAIN_CLIPS_PER_CAMERA_CLASS = 3
TEST_CLIPS_PER_CAMERA_CLASS = 2
TEST_DATES = {"2025-10-25", "2025-10-26"}
RANDOM_SEED = 42
MAX_UNLABELED_VIDEOS_PER_CAMERA = 0  # Set >0 only for a separate exploratory run.

VEHICLE_CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
COUNTING_LINES = {
    "default": ((0.05, 0.50), (0.95, 0.50)),
    # Calibrate each line against an annotated frame before comparing cameras.
    # "normanniles1": ((0.05, 0.52), (0.95, 0.52)),
}
CAMERA_FOLDERS = {
    "camera1": os.path.join(DRIVE_BASE, "normanniles1"),
    "camera2": os.path.join(DRIVE_BASE, "normanniles2"),
    "camera3": os.path.join(DRIVE_BASE, "normanniles3"),
    "camera4": os.path.join(DRIVE_BASE, "normanniles4"),
    "added": os.path.join(DRIVE_BASE, "added"),
}
for directory in (ANNOTATION_DIR, RUN_LOG_DIR, JOURNAL_EXPORT_DIR):
    os.makedirs(directory, exist_ok=True)

def make_video_key(video_path):
    """Return the Train.csv-compatible relative path."""
    rel_path = os.path.relpath(video_path, DRIVE_BASE).replace("\\", "/")
    return os.path.basename(video_path) if rel_path.startswith("..") else rel_path

all_videos = {}
for cam_name, cam_path in CAMERA_FOLDERS.items():
    all_videos[cam_name] = sorted(glob(os.path.join(cam_path, "**", "*.mp4"), recursive=True)) if os.path.isdir(cam_path) else []
    print(f"{cam_name}: {len(all_videos[cam_name])} videos")
print(f"Total videos: {sum(map(len, all_videos.values()))}; model: {YOLO_MODEL_NAME}")
print(f"New feature output (old CSV retained): {FEATURES_CSV_PATH}")

train_df = pd.read_csv(TRAIN_CSV_PATH) if os.path.isfile(TRAIN_CSV_PATH) else None
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH) if os.path.isfile(SAMPLE_SUBMISSION_PATH) else None
print(f"Train labels: {len(train_df) if train_df is not None else 'not found'}")
if train_df is not None:
    display(train_df.head())

# Frozen selection from the user's completed YOLO11 v2 export.
REFERENCE_EXPORT_NAME = 'Journal_Results_Export_v2-20261020T090346Z-1-001.zip'
REFERENCE_SELECTION_SHA256 = '674d6de59ac56a0fd76730e00d46135330f08c70704be45d2d3a0197eb038e01'
REFERENCE_PAIRS_CSV = 'responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase,source_time,segment_end,target_time,target_video_path,next_enter_rating,gap_sec,horizon_sec,source_date,camera_id,split\n0okHaZimMsjtnvgP8NiHuAw,Norman Niles #1,time_segment_3996_Norman Niles #1_congestion_enter_rating,time_segment_3996_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-25-13-21-45.mp4,2025-10-25 13:21:45,2025-10-25 13:21:44,2025-10-25 13:22:43,2025-10-25,none,free flowing,free flowing,3996,train,2025-10-25 13:21:44,2025-10-25 13:22:43,2025-10-25 13:22:44,normanniles1/normanniles1_2025-10-25-13-22-45.mp4,light delay,1.0,60.0,2025-10-25,normanniles1,test\nD4sHabtf0JPc2g_ys8igDQ,Norman Niles #1,time_segment_4044_Norman Niles #1_congestion_enter_rating,time_segment_4044_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-25-14-09-45.mp4,2025-10-25 14:09:45,2025-10-25 14:09:44,2025-10-25 14:10:43,2025-10-25,medium,free flowing,free flowing,4044,train,2025-10-25 14:09:44,2025-10-25 14:10:43,2025-10-25 14:10:44,normanniles1/normanniles1_2025-10-25-14-10-45.mp4,heavy delay,1.0,60.0,2025-10-25,normanniles1,test\nLosHacnDKrOv3NoPq7G46QE,Norman Niles #1,time_segment_4218_Norman Niles #1_congestion_enter_rating,time_segment_4218_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-25-17-03-45.mp4,2025-10-25 17:03:45,2025-10-25 17:03:44,2025-10-25 17:04:43,2025-10-25,none,free flowing,free flowing,4218,train,2025-10-25 17:03:44,2025-10-25 17:04:43,2025-10-25 17:04:44,normanniles1/normanniles1_2025-10-25-17-04-45.mp4,free flowing,1.0,60.0,2025-10-25,normanniles1,test\nzIkHaa_DN-K2nvgPsZGZ4Aw,Norman Niles #1,time_segment_4261_Norman Niles #1_congestion_enter_rating,time_segment_4261_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-25-17-46-45.mp4,2025-10-25 17:46:45,2025-10-25 17:46:44,2025-10-25 17:47:43,2025-10-25,medium,free flowing,free flowing,4261,train,2025-10-25 17:46:44,2025-10-25 17:47:43,2025-10-25 17:47:44,normanniles1/normanniles1_2025-10-25-17-47-45.mp4,light delay,1.0,60.0,2025-10-25,normanniles1,test\nCI0HacalLMjtnvgP8NiHuAw,Norman Niles #1,time_segment_4270_Norman Niles #1_congestion_enter_rating,time_segment_4270_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-25-17-55-45.mp4,2025-10-25 17:55:45,2025-10-25 17:55:44,2025-10-25 17:56:43,2025-10-25,medium,light delay,free flowing,4270,train,2025-10-25 17:55:44,2025-10-25 17:56:43,2025-10-25 17:56:44,normanniles1/normanniles1_2025-10-25-17-56-45.mp4,heavy delay,1.0,60.0,2025-10-25,normanniles1,test\nGY0HaeKjDrmfnvgPoaTSqQ0,Norman Niles #1,time_segment_4705_Norman Niles #1_congestion_enter_rating,time_segment_4705_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-26-13-14-45.mp4,2025-10-26 13:14:45,2025-10-26 13:14:33,2025-10-26 13:15:41,2025-10-26,none,free flowing,free flowing,4705,train,2025-10-26 13:14:33,2025-10-26 13:15:41,2025-10-26 13:15:42,normanniles1/normanniles1_2025-10-26-13-15-45.mp4,moderate delay,1.0,69.0,2025-10-26,normanniles1,test\nA4oHafLsN6OdmecPmP768Ag,Norman Niles #1,time_segment_4944_Norman Niles #1_congestion_enter_rating,time_segment_4944_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-26-17-16-45.mp4,2025-10-26 17:16:45,2025-10-26 17:16:39,2025-10-26 17:17:41,2025-10-26,none,free flowing,free flowing,4944,train,2025-10-26 17:16:39,2025-10-26 17:17:41,2025-10-26 17:17:42,normanniles1/normanniles1_2025-10-26-17-17-45.mp4,moderate delay,1.0,63.0,2025-10-26,normanniles1,test\nAYoHaZvkKdm7nvgPq8e12AU,Norman Niles #1,time_segment_4967_Norman Niles #1_congestion_enter_rating,time_segment_4967_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-26-17-39-45.mp4,2025-10-26 17:39:45,2025-10-26 17:39:33,2025-10-26 17:40:42,2025-10-26,none,light delay,free flowing,4967,train,2025-10-26 17:39:33,2025-10-26 17:40:42,2025-10-26 17:40:42,normanniles1/normanniles1_2025-10-26-17-40-45.mp4,free flowing,0.0,69.0,2025-10-26,normanniles1,test\nOIsHab_pN6-vnvgPjoXZqA0,Norman Niles #2,time_segment_3959_Norman Niles #2_congestion_enter_rating,time_segment_3959_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-25-12-44-45.mp4,2025-10-25 12:44:45,2025-10-25 12:44:47,2025-10-25 12:45:44,2025-10-25,low,light delay,free flowing,3959,train,2025-10-25 12:44:47,2025-10-25 12:45:44,2025-10-25 12:45:45,normanniles2/normanniles2_2025-10-25-12-45-45.mp4,light delay,1.0,58.0,2025-10-25,normanniles2,test\n04kHae3xC7mfnvgPoaTSqQ0,Norman Niles #2,time_segment_3983_Norman Niles #2_congestion_enter_rating,time_segment_3983_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-25-13-08-45.mp4,2025-10-25 13:08:45,2025-10-25 13:08:47,2025-10-25 13:09:46,2025-10-25,medium,light delay,free flowing,3983,train,2025-10-25 13:08:47,2025-10-25 13:09:46,2025-10-25 13:09:47,normanniles2/normanniles2_2025-10-25-13-09-45.mp4,moderate delay,1.0,60.0,2025-10-25,normanniles2,test\n7YoHafjUL5KdmecPt_3RiQ4,Norman Niles #2,time_segment_4058_Norman Niles #2_congestion_enter_rating,time_segment_4058_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-25-14-23-45.mp4,2025-10-25 14:23:45,2025-10-25 14:23:47,2025-10-25 14:24:46,2025-10-25,high,moderate delay,light delay,4058,train,2025-10-25 14:23:47,2025-10-25 14:24:46,2025-10-25 14:24:47,normanniles2/normanniles2_2025-10-25-14-24-45.mp4,moderate delay,1.0,60.0,2025-10-25,normanniles2,test\nBIoHafCDF9m7nvgPq8e12AU,Norman Niles #2,time_segment_4204_Norman Niles #2_congestion_enter_rating,time_segment_4204_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-25-16-49-45.mp4,2025-10-25 16:49:45,2025-10-25 16:49:48,2025-10-25 16:50:47,2025-10-25,low,moderate delay,free flowing,4204,train,2025-10-25 16:49:48,2025-10-25 16:50:47,2025-10-25 16:50:48,normanniles2/normanniles2_2025-10-25-16-50-45.mp4,heavy delay,1.0,60.0,2025-10-25,normanniles2,test\n7YkHabTwMq-vnvgPjoXZqA0,Norman Niles #2,time_segment_4259_Norman Niles #2_congestion_enter_rating,time_segment_4259_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-25-17-44-45.mp4,2025-10-25 17:44:45,2025-10-25 17:44:48,2025-10-25 17:45:48,2025-10-25,low,free flowing,free flowing,4259,train,2025-10-25 17:44:48,2025-10-25 17:45:48,2025-10-25 17:45:48,normanniles2/normanniles2_2025-10-25-17-45-45.mp4,heavy delay,0.0,60.0,2025-10-25,normanniles2,test\nLI0HafNA2IPY-A_IpprJDA,Norman Niles #2,time_segment_4304_Norman Niles #2_congestion_enter_rating,time_segment_4304_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-26-06-29-45.mp4,2025-10-26 06:29:45,2025-10-26 06:29:46,2025-10-26 06:30:45,2025-10-26,none,free flowing,free flowing,4304,train,2025-10-26 06:29:46,2025-10-26 06:30:45,2025-10-26 06:30:46,normanniles2/normanniles2_2025-10-26-06-30-45.mp4,free flowing,1.0,60.0,2025-10-26,normanniles2,test\n04kHaaGCDtiD2PgPyKaayQw,Norman Niles #2,time_segment_4755_Norman Niles #2_congestion_enter_rating,time_segment_4755_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-26-14-05-45.mp4,2025-10-26 14:05:45,2025-10-26 14:05:47,2025-10-26 14:06:46,2025-10-26,none,free flowing,free flowing,4755,train,2025-10-26 14:05:47,2025-10-26 14:06:46,2025-10-26 14:06:47,normanniles2/normanniles2_2025-10-26-14-06-45.mp4,light delay,1.0,60.0,2025-10-26,normanniles2,test\nI4sHaeHzDbXOnvgP3YW5kQs,Norman Niles #2,time_segment_4812_Norman Niles #2_congestion_enter_rating,time_segment_4812_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-26-15-02-45.mp4,2025-10-26 15:02:45,2025-10-26 15:02:47,2025-10-26 15:03:46,2025-10-26,low,moderate delay,free flowing,4812,train,2025-10-26 15:02:47,2025-10-26 15:03:46,2025-10-26 15:03:47,normanniles2/normanniles2_2025-10-26-15-03-45.mp4,free flowing,1.0,60.0,2025-10-26,normanniles2,test\n3owHaZUp4rae-A-xkZngDA,Norman Niles #3,time_segment_3773_Norman Niles #3_congestion_enter_rating,time_segment_3773_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-25-09-33-45.mp4,2025-10-25 09:33:45,2025-10-25 09:33:44,2025-10-25 09:34:43,2025-10-25,none,free flowing,free flowing,3773,train,2025-10-25 09:33:44,2025-10-25 09:34:43,2025-10-25 09:34:44,normanniles3/normanniles3_2025-10-25-09-34-45.mp4,free flowing,1.0,60.0,2025-10-25,normanniles3,test\n2owHacb3OLOv3NoPq7G46QE,Norman Niles #3,time_segment_3867_Norman Niles #3_congestion_enter_rating,time_segment_3867_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-25-11-07-45.mp4,2025-10-25 11:07:45,2025-10-25 11:07:44,2025-10-25 11:08:43,2025-10-25,low,free flowing,free flowing,3867,train,2025-10-25 11:07:44,2025-10-25 11:08:43,2025-10-25 11:08:44,normanniles3/normanniles3_2025-10-25-11-08-45.mp4,light delay,1.0,60.0,2025-10-25,normanniles3,test\n0okHab2sHdCT3NoP8rPIoA0,Norman Niles #3,time_segment_3975_Norman Niles #3_congestion_enter_rating,time_segment_3975_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-25-13-00-45.mp4,2025-10-25 13:00:45,2025-10-25 13:00:39,2025-10-25 13:01:39,2025-10-25,medium,heavy delay,heavy delay,3975,train,2025-10-25 13:00:39,2025-10-25 13:01:39,2025-10-25 13:01:39,normanniles3/normanniles3_2025-10-25-13-01-45.mp4,heavy delay,0.0,60.0,2025-10-25,normanniles3,test\naYwHaeb_EsjtnvgP8NiHuAw,Norman Niles #3,time_segment_4169_Norman Niles #3_congestion_enter_rating,time_segment_4169_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-25-16-14-45.mp4,2025-10-25 16:14:45,2025-10-25 16:14:43,2025-10-25 16:15:42,2025-10-25,none,moderate delay,free flowing,4169,train,2025-10-25 16:14:43,2025-10-25 16:15:42,2025-10-25 16:15:44,normanniles3/normanniles3_2025-10-25-16-15-45.mp4,moderate delay,2.0,61.0,2025-10-25,normanniles3,test\n0okHaanQLdiNmecPivmHgQ0,Norman Niles #3,time_segment_4235_Norman Niles #3_congestion_enter_rating,time_segment_4235_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-25-17-20-45.mp4,2025-10-25 17:20:45,2025-10-25 17:20:44,2025-10-25 17:21:43,2025-10-25,medium,free flowing,free flowing,4235,train,2025-10-25 17:20:44,2025-10-25 17:21:43,2025-10-25 17:21:44,normanniles3/normanniles3_2025-10-25-17-21-45.mp4,heavy delay,1.0,60.0,2025-10-25,normanniles3,test\n9YkHaYn-A6qpmecP5vCV0AQ,Norman Niles #3,time_segment_4706_Norman Niles #3_congestion_enter_rating,time_segment_4706_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-26-13-15-45.mp4,2025-10-26 13:15:45,2025-10-26 13:15:43,2025-10-26 13:16:43,2025-10-26,low,free flowing,free flowing,4706,train,2025-10-26 13:15:43,2025-10-26 13:16:43,2025-10-26 13:16:43,normanniles3/normanniles3_2025-10-26-13-16-45.mp4,moderate delay,0.0,60.0,2025-10-26,normanniles3,test\n8IwHaZizLrXOnvgP3YW5kQs,Norman Niles #3,time_segment_4850_Norman Niles #3_congestion_enter_rating,time_segment_4850_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-26-15-40-45.mp4,2025-10-26 15:40:45,2025-10-26 15:40:43,2025-10-26 15:41:42,2025-10-26,none,free flowing,free flowing,4850,train,2025-10-26 15:40:43,2025-10-26 15:41:42,2025-10-26 15:41:43,normanniles3/normanniles3_2025-10-26-15-41-45.mp4,light delay,1.0,60.0,2025-10-26,normanniles3,test\n0okHacTEEdm7nvgPq8e12AU,Norman Niles #3,time_segment_4958_Norman Niles #3_congestion_enter_rating,time_segment_4958_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-26-17-30-45.mp4,2025-10-26 17:30:45,2025-10-26 17:30:39,2025-10-26 17:31:40,2025-10-26,low,light delay,free flowing,4958,train,2025-10-26 17:30:39,2025-10-26 17:31:40,2025-10-26 17:31:41,normanniles3/normanniles3_2025-10-26-17-31-45.mp4,free flowing,1.0,62.0,2025-10-26,normanniles3,test\nzIkHadyrOOK2nvgPsZGZ4Aw,Norman Niles #4,time_segment_3925_Norman Niles #4_congestion_enter_rating,time_segment_3925_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-25-12-05-45.mp4,2025-10-25 12:05:45,2025-10-25 12:05:44,2025-10-25 12:06:43,2025-10-25,low,free flowing,free flowing,3925,train,2025-10-25 12:05:44,2025-10-25 12:06:43,2025-10-25 12:06:44,normanniles4/normanniles4_2025-10-25-12-06-45.mp4,light delay,1.0,60.0,2025-10-25,normanniles4,test\nI4sHaYmFD8jtnvgP8NiHuAw,Norman Niles #4,time_segment_4026_Norman Niles #4_congestion_enter_rating,time_segment_4026_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-25-13-51-45.mp4,2025-10-25 13:51:45,2025-10-25 13:51:44,2025-10-25 13:52:43,2025-10-25,low,moderate delay,free flowing,4026,train,2025-10-25 13:51:44,2025-10-25 13:52:43,2025-10-25 13:52:44,normanniles4/normanniles4_2025-10-25-13-52-45.mp4,heavy delay,1.0,60.0,2025-10-25,normanniles4,test\nB4oHaaWHL6SXmecP9PbSyA8,Norman Niles #4,time_segment_4040_Norman Niles #4_congestion_enter_rating,time_segment_4040_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-25-14-05-45.mp4,2025-10-25 14:05:45,2025-10-25 14:05:44,2025-10-25 14:06:43,2025-10-25,medium,heavy delay,free flowing,4040,train,2025-10-25 14:05:44,2025-10-25 14:06:43,2025-10-25 14:06:44,normanniles4/normanniles4_2025-10-25-14-06-45.mp4,moderate delay,1.0,60.0,2025-10-25,normanniles4,test\n24wHae22H7zA698P_KuO8AM,Norman Niles #4,time_segment_4063_Norman Niles #4_congestion_enter_rating,time_segment_4063_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-25-14-28-45.mp4,2025-10-25 14:28:45,2025-10-25 14:28:44,2025-10-25 14:29:43,2025-10-25,low,heavy delay,free flowing,4063,train,2025-10-25 14:28:44,2025-10-25 14:29:43,2025-10-25 14:29:44,normanniles4/normanniles4_2025-10-25-14-29-45.mp4,heavy delay,1.0,60.0,2025-10-25,normanniles4,test\n7IkHacryO9P_ld8Piv3HsQo,Norman Niles #4,time_segment_4220_Norman Niles #4_congestion_enter_rating,time_segment_4220_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-25-17-05-45.mp4,2025-10-25 17:05:45,2025-10-25 17:05:43,2025-10-25 17:06:42,2025-10-25,low,free flowing,free flowing,4220,train,2025-10-25 17:05:43,2025-10-25 17:06:42,2025-10-25 17:06:43,normanniles4/normanniles4_2025-10-25-17-06-45.mp4,moderate delay,1.0,60.0,2025-10-25,normanniles4,test\n74kHafTxCq-vnvgPjoXZqA0,Norman Niles #4,time_segment_4298_Norman Niles #4_congestion_enter_rating,time_segment_4298_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-26-06-23-45.mp4,2025-10-26 06:23:45,2025-10-26 06:23:42,2025-10-26 06:24:43,2025-10-26,none,free flowing,free flowing,4298,train,2025-10-26 06:23:42,2025-10-26 06:24:43,2025-10-26 06:24:44,normanniles4/normanniles4_2025-10-26-06-24-45.mp4,free flowing,1.0,62.0,2025-10-26,normanniles4,test\nSIsHaYzhE-aQ2PgPqvKU8As,Norman Niles #4,time_segment_4822_Norman Niles #4_congestion_enter_rating,time_segment_4822_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-26-15-12-45.mp4,2025-10-26 15:12:45,2025-10-26 15:12:44,2025-10-26 15:13:43,2025-10-26,medium,free flowing,free flowing,4822,train,2025-10-26 15:12:44,2025-10-26 15:13:43,2025-10-26 15:13:44,normanniles4/normanniles4_2025-10-26-15-13-45.mp4,free flowing,1.0,60.0,2025-10-26,normanniles4,test\nBIoHadL_KqCVmecP_oCp4QE,Norman Niles #4,time_segment_4885_Norman Niles #4_congestion_enter_rating,time_segment_4885_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-26-16-15-45.mp4,2025-10-26 16:15:45,2025-10-26 16:15:44,2025-10-26 16:16:43,2025-10-26,none,light delay,free flowing,4885,train,2025-10-26 16:15:44,2025-10-26 16:16:43,2025-10-26 16:16:44,normanniles4/normanniles4_2025-10-26-16-16-45.mp4,light delay,1.0,60.0,2025-10-26,normanniles4,test\nBIoHaYuiN-K2nvgPsZGZ4Aw,Norman Niles #1,time_segment_157_Norman Niles #1_congestion_enter_rating,time_segment_157_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-20-08-37-45.mp4,2025-10-20 08:37:45,2025-10-20 08:37:45,2025-10-20 08:38:44,2025-10-20,none,free flowing,free flowing,157,train,2025-10-20 08:37:45,2025-10-20 08:38:44,2025-10-20 08:38:45,normanniles1/normanniles1_2025-10-20-08-38-45.mp4,free flowing,1.0,60.0,2025-10-20,normanniles1,train\n2IwHabHwK-K2nvgPsZGZ4Aw,Norman Niles #1,time_segment_316_Norman Niles #1_congestion_enter_rating,time_segment_316_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-20-11-18-45.mp4,2025-10-20 11:18:45,2025-10-20 11:18:45,2025-10-20 11:19:44,2025-10-20,low,moderate delay,free flowing,316,train,2025-10-20 11:18:45,2025-10-20 11:19:44,2025-10-20 11:19:45,normanniles1/normanniles1_2025-10-20-11-19-45.mp4,heavy delay,1.0,60.0,2025-10-20,normanniles1,train\nEIsHaaeaNdm7nvgPq8e12AU,Norman Niles #1,time_segment_331_Norman Niles #1_congestion_enter_rating,time_segment_331_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-20-11-33-45.mp4,2025-10-20 11:33:45,2025-10-20 11:33:44,2025-10-20 11:34:44,2025-10-20,low,free flowing,free flowing,331,train,2025-10-20 11:33:44,2025-10-20 11:34:44,2025-10-20 11:34:44,normanniles1/normanniles1_2025-10-20-11-34-45.mp4,light delay,0.0,60.0,2025-10-20,normanniles1,train\n2YwHaei4IbmfnvgPoaTSqQ0,Norman Niles #1,time_segment_478_Norman Niles #1_congestion_enter_rating,time_segment_478_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-20-14-02-45.mp4,2025-10-20 14:02:45,2025-10-20 14:02:44,2025-10-20 14:03:43,2025-10-20,none,free flowing,free flowing,478,train,2025-10-20 14:02:44,2025-10-20 14:03:43,2025-10-20 14:03:44,normanniles1/normanniles1_2025-10-20-14-03-45.mp4,light delay,1.0,60.0,2025-10-20,normanniles1,train\naYwHaZifINiD2PgPyKaayQw,Norman Niles #1,time_segment_838_Norman Niles #1_congestion_enter_rating,time_segment_838_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-21-08-05-45.mp4,2025-10-21 08:05:45,2025-10-21 08:05:45,2025-10-21 08:06:44,2025-10-21,none,free flowing,free flowing,838,train,2025-10-21 08:05:45,2025-10-21 08:06:44,2025-10-21 08:06:45,normanniles1/normanniles1_2025-10-21-08-06-45.mp4,free flowing,1.0,60.0,2025-10-21,normanniles1,train\nAIsHacP5GtCT3NoP8rPIoA0,Norman Niles #1,time_segment_1012_Norman Niles #1_congestion_enter_rating,time_segment_1012_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-21-10-59-45.mp4,2025-10-21 10:59:45,2025-10-21 10:59:44,2025-10-21 11:00:43,2025-10-21,medium,heavy delay,moderate delay,1012,train,2025-10-21 10:59:44,2025-10-21 11:00:43,2025-10-21 11:00:45,normanniles1/normanniles1_2025-10-21-11-00-45.mp4,moderate delay,2.0,61.0,2025-10-21,normanniles1,train\n0okHaZ-pO5__698PweWaiQ4,Norman Niles #1,time_segment_1395_Norman Niles #1_congestion_enter_rating,time_segment_1395_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-21-17-31-45.mp4,2025-10-21 17:31:45,2025-10-21 17:31:44,2025-10-21 17:32:43,2025-10-21,none,moderate delay,free flowing,1395,train,2025-10-21 17:31:44,2025-10-21 17:32:43,2025-10-21 17:32:44,normanniles1/normanniles1_2025-10-21-17-32-45.mp4,heavy delay,1.0,60.0,2025-10-21,normanniles1,train\nAooHadiYMeaQ2PgPqvKU8As,Norman Niles #1,time_segment_2323_Norman Niles #1_congestion_enter_rating,time_segment_2323_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-23-09-03-45.mp4,2025-10-23 09:03:45,2025-10-23 09:03:44,2025-10-23 09:04:43,2025-10-23,none,free flowing,free flowing,2323,train,2025-10-23 09:03:44,2025-10-23 09:04:43,2025-10-23 09:04:44,normanniles1/normanniles1_2025-10-23-09-04-45.mp4,free flowing,1.0,60.0,2025-10-23,normanniles1,train\nIosHaY3vOp__698PweWaiQ4,Norman Niles #1,time_segment_2400_Norman Niles #1_congestion_enter_rating,time_segment_2400_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-23-10-24-45.mp4,2025-10-23 10:24:45,2025-10-23 10:24:44,2025-10-23 10:25:43,2025-10-23,none,moderate delay,free flowing,2400,train,2025-10-23 10:24:44,2025-10-23 10:25:43,2025-10-23 10:25:44,normanniles1/normanniles1_2025-10-23-10-25-45.mp4,moderate delay,1.0,60.0,2025-10-23,normanniles1,train\nAIoHab-9Bb2Flb4Pkc-IgAo,Norman Niles #1,time_segment_2527_Norman Niles #1_congestion_enter_rating,time_segment_2527_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-23-12-35-45.mp4,2025-10-23 12:35:45,2025-10-23 12:35:44,2025-10-23 12:36:44,2025-10-23,low,free flowing,free flowing,2527,train,2025-10-23 12:35:44,2025-10-23 12:36:44,2025-10-23 12:36:44,normanniles1/normanniles1_2025-10-23-12-36-45.mp4,light delay,0.0,60.0,2025-10-23,normanniles1,train\nHo0Hafe6Bp__698PweWaiQ4,Norman Niles #1,time_segment_2842_Norman Niles #1_congestion_enter_rating,time_segment_2842_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-23-17-50-45.mp4,2025-10-23 17:50:45,2025-10-23 17:50:44,2025-10-23 17:51:44,2025-10-23,low,moderate delay,free flowing,2842,train,2025-10-23 17:50:44,2025-10-23 17:51:44,2025-10-23 17:51:44,normanniles1/normanniles1_2025-10-23-17-51-45.mp4,heavy delay,0.0,60.0,2025-10-23,normanniles1,train\nIIsHaZ_mLNm7nvgPq8e12AU,Norman Niles #1,time_segment_3439_Norman Niles #1_congestion_enter_rating,time_segment_3439_Norman Niles #1_congestion_exit_rating,normanniles1/normanniles1_2025-10-24-15-54-45.mp4,2025-10-24 15:54:45,2025-10-24 15:54:43,2025-10-24 15:55:43,2025-10-24,low,heavy delay,free flowing,3439,train,2025-10-24 15:54:43,2025-10-24 15:55:43,2025-10-24 15:55:44,normanniles1/normanniles1_2025-10-24-15-55-45.mp4,moderate delay,1.0,61.0,2025-10-24,normanniles1,train\n7okHaenCHJKdmecPt_3RiQ4,Norman Niles #2,time_segment_667_Norman Niles #2_congestion_enter_rating,time_segment_667_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-20-17-13-45.mp4,2025-10-20 17:13:45,2025-10-20 17:13:47,2025-10-20 17:14:46,2025-10-20,medium,free flowing,free flowing,667,train,2025-10-20 17:13:47,2025-10-20 17:14:46,2025-10-20 17:14:46,normanniles2/normanniles2_2025-10-20-17-14-45.mp4,light delay,0.0,59.0,2025-10-20,normanniles2,train\n_owHabawOq-vnvgPjoXZqA0,Norman Niles #2,time_segment_779_Norman Niles #2_congestion_enter_rating,time_segment_779_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-21-07-06-45.mp4,2025-10-21 07:06:45,2025-10-21 07:06:46,2025-10-21 07:07:45,2025-10-21,none,light delay,free flowing,779,train,2025-10-21 07:06:46,2025-10-21 07:07:45,2025-10-21 07:07:46,normanniles2/normanniles2_2025-10-21-07-07-45.mp4,free flowing,1.0,60.0,2025-10-21,normanniles2,train\nLIsHadiWHNm7nvgPq8e12AU,Norman Niles #2,time_segment_1047_Norman Niles #2_congestion_enter_rating,time_segment_1047_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-21-11-34-45.mp4,2025-10-21 11:34:45,2025-10-21 11:34:47,2025-10-21 11:35:46,2025-10-21,medium,light delay,free flowing,1047,train,2025-10-21 11:34:47,2025-10-21 11:35:46,2025-10-21 11:35:47,normanniles2/normanniles2_2025-10-21-11-35-45.mp4,moderate delay,1.0,60.0,2025-10-21,normanniles2,train\n0YkHabDKIK-vnvgPjoXZqA0,Norman Niles #2,time_segment_1392_Norman Niles #2_congestion_enter_rating,time_segment_1392_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-21-17-28-45.mp4,2025-10-21 17:28:45,2025-10-21 17:28:48,2025-10-21 17:29:47,2025-10-21,low,moderate delay,free flowing,1392,train,2025-10-21 17:28:48,2025-10-21 17:29:47,2025-10-21 17:29:48,normanniles2/normanniles2_2025-10-21-17-29-45.mp4,heavy delay,1.0,60.0,2025-10-21,normanniles2,train\nDIsHaa-OC7Ov3NoPq7G46QE,Norman Niles #2,time_segment_1735_Norman Niles #2_congestion_enter_rating,time_segment_1735_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-22-11-11-45.mp4,2025-10-22 11:11:45,2025-10-22 11:11:46,2025-10-22 11:12:46,2025-10-22,medium,moderate delay,free flowing,1735,train,2025-10-22 11:11:46,2025-10-22 11:12:46,2025-10-22 11:12:46,normanniles2/normanniles2_2025-10-22-11-12-45.mp4,light delay,0.0,60.0,2025-10-22,normanniles2,train\nSIsHaeH_K-K2nvgPsZGZ4Aw,Norman Niles #2,time_segment_2045_Norman Niles #2_congestion_enter_rating,time_segment_2045_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-22-16-24-45.mp4,2025-10-22 16:24:45,2025-10-22 16:24:47,2025-10-22 16:25:46,2025-10-22,high,light delay,free flowing,2045,train,2025-10-22 16:24:47,2025-10-22 16:25:46,2025-10-22 16:25:47,normanniles2/normanniles2_2025-10-22-16-25-45.mp4,free flowing,1.0,60.0,2025-10-22,normanniles2,train\nAYoHadX2KqCVmecP_oCp4QE,Norman Niles #2,time_segment_2079_Norman Niles #2_congestion_enter_rating,time_segment_2079_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-22-16-58-45.mp4,2025-10-22 16:58:45,2025-10-22 16:58:47,2025-10-22 16:59:47,2025-10-22,none,moderate delay,free flowing,2079,train,2025-10-22 16:58:47,2025-10-22 16:59:47,2025-10-22 16:59:47,normanniles2/normanniles2_2025-10-22-16-59-45.mp4,heavy delay,0.0,60.0,2025-10-22,normanniles2,train\nPYoHaeqlM9mqmecPxebXqQE,Norman Niles #2,time_segment_2730_Norman Niles #2_congestion_enter_rating,time_segment_2730_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-23-15-58-45.mp4,2025-10-23 15:58:45,2025-10-23 15:58:45,2025-10-23 15:59:44,2025-10-23,medium,moderate delay,free flowing,2730,train,2025-10-23 15:58:45,2025-10-23 15:59:44,2025-10-23 15:59:45,normanniles2/normanniles2_2025-10-23-15-59-45.mp4,heavy delay,1.0,60.0,2025-10-23,normanniles2,train\nBooHaY3qIceTmecPu8rsgQE,Norman Niles #2,time_segment_3159_Norman Niles #2_congestion_enter_rating,time_segment_3159_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-24-11-07-45.mp4,2025-10-24 11:07:45,2025-10-24 11:07:46,2025-10-24 11:08:45,2025-10-24,medium,light delay,free flowing,3159,train,2025-10-24 11:07:46,2025-10-24 11:08:45,2025-10-24 11:08:46,normanniles2/normanniles2_2025-10-24-11-08-45.mp4,moderate delay,1.0,60.0,2025-10-24,normanniles2,train\n24wHafcw0JPc2g_ys8igDQ,Norman Niles #2,time_segment_3281_Norman Niles #2_congestion_enter_rating,time_segment_3281_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-24-13-10-45.mp4,2025-10-24 13:10:45,2025-10-24 13:10:47,2025-10-24 13:11:46,2025-10-24,low,light delay,free flowing,3281,train,2025-10-24 13:10:47,2025-10-24 13:11:46,2025-10-24 13:11:47,normanniles2/normanniles2_2025-10-24-13-11-45.mp4,moderate delay,1.0,60.0,2025-10-24,normanniles2,train\nS4sHaZTGG6-vnvgPjoXZqA0,Norman Niles #2,time_segment_3390_Norman Niles #2_congestion_enter_rating,time_segment_3390_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-24-15-05-45.mp4,2025-10-24 15:05:45,2025-10-24 15:05:44,2025-10-24 15:06:43,2025-10-24,medium,moderate delay,free flowing,3390,train,2025-10-24 15:05:44,2025-10-24 15:06:43,2025-10-24 15:06:44,normanniles2/normanniles2_2025-10-24-15-06-45.mp4,free flowing,1.0,60.0,2025-10-24,normanniles2,train\nIY0HafW4JZ__698PweWaiQ4,Norman Niles #2,time_segment_3535_Norman Niles #2_congestion_enter_rating,time_segment_3535_Norman Niles #2_congestion_exit_rating,normanniles2/normanniles2_2025-10-24-17-35-45.mp4,2025-10-24 17:35:45,2025-10-24 17:35:44,2025-10-24 17:36:43,2025-10-24,low,free flowing,free flowing,3535,train,2025-10-24 17:35:44,2025-10-24 17:36:43,2025-10-24 17:36:44,normanniles2/normanniles2_2025-10-24-17-36-45.mp4,light delay,1.0,60.0,2025-10-24,normanniles2,train\n0okHaf-YCLXOnvgP3YW5kQs,Norman Niles #3,time_segment_404_Norman Niles #3_congestion_enter_rating,time_segment_404_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-20-12-47-45.mp4,2025-10-20 12:47:45,2025-10-20 12:47:43,2025-10-20 12:48:43,2025-10-20,none,free flowing,free flowing,404,train,2025-10-20 12:47:43,2025-10-20 12:48:43,2025-10-20 12:48:43,normanniles3/normanniles3_2025-10-20-12-48-45.mp4,moderate delay,0.0,60.0,2025-10-20,normanniles3,train\nBY0HafnULr-envgP6uGTmAE,Norman Niles #3,time_segment_540_Norman Niles #3_congestion_enter_rating,time_segment_540_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-20-15-04-45.mp4,2025-10-20 15:04:45,2025-10-20 15:04:42,2025-10-20 15:05:41,2025-10-20,none,light delay,free flowing,540,train,2025-10-20 15:04:42,2025-10-20 15:05:41,2025-10-20 15:05:42,normanniles3/normanniles3_2025-10-20-15-05-45.mp4,light delay,1.0,60.0,2025-10-20,normanniles3,train\nA40HadboIOaQ2PgPqvKU8As,Norman Niles #3,time_segment_583_Norman Niles #3_congestion_enter_rating,time_segment_583_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-20-15-47-45.mp4,2025-10-20 15:47:45,2025-10-20 15:47:42,2025-10-20 15:48:41,2025-10-20,none,free flowing,free flowing,583,train,2025-10-20 15:47:42,2025-10-20 15:48:41,2025-10-20 15:48:42,normanniles3/normanniles3_2025-10-20-15-48-45.mp4,moderate delay,1.0,60.0,2025-10-20,normanniles3,train\nNYsHaf7kObOv3NoPq7G46QE,Norman Niles #3,time_segment_662_Norman Niles #3_congestion_enter_rating,time_segment_662_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-20-17-08-45.mp4,2025-10-20 17:08:45,2025-10-20 17:08:42,2025-10-20 17:09:41,2025-10-20,none,free flowing,free flowing,662,train,2025-10-20 17:08:42,2025-10-20 17:09:41,2025-10-20 17:09:42,normanniles3/normanniles3_2025-10-20-17-09-45.mp4,free flowing,1.0,60.0,2025-10-20,normanniles3,train\nDIsHaZ-RK9iD2PgPyKaayQw,Norman Niles #3,time_segment_799_Norman Niles #3_congestion_enter_rating,time_segment_799_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-21-07-26-45.mp4,2025-10-21 07:26:45,2025-10-21 07:26:44,2025-10-21 07:27:43,2025-10-21,none,free flowing,free flowing,799,train,2025-10-21 07:26:44,2025-10-21 07:27:43,2025-10-21 07:27:44,normanniles3/normanniles3_2025-10-21-07-27-45.mp4,free flowing,1.0,60.0,2025-10-21,normanniles3,train\nXIoHaYeMDOK2nvgPsZGZ4Aw,Norman Niles #3,time_segment_1351_Norman Niles #3_congestion_enter_rating,time_segment_1351_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-21-16-47-45.mp4,2025-10-21 16:47:45,2025-10-21 16:47:43,2025-10-21 16:48:42,2025-10-21,none,free flowing,free flowing,1351,train,2025-10-21 16:47:43,2025-10-21 16:48:42,2025-10-21 16:48:43,normanniles3/normanniles3_2025-10-21-16-48-45.mp4,heavy delay,1.0,60.0,2025-10-21,normanniles3,train\n04kHaciaL6HXnvgP76fxyQI,Norman Niles #3,time_segment_1750_Norman Niles #3_congestion_enter_rating,time_segment_1750_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-22-11-26-45.mp4,2025-10-22 11:26:45,2025-10-22 11:26:44,2025-10-22 11:27:43,2025-10-22,low,moderate delay,free flowing,1750,train,2025-10-22 11:26:44,2025-10-22 11:27:43,2025-10-22 11:27:44,normanniles3/normanniles3_2025-10-22-11-27-45.mp4,heavy delay,1.0,60.0,2025-10-22,normanniles3,train\nv4wHaaKMI6qpmecP5vCV0AQ,Norman Niles #3,time_segment_2132_Norman Niles #3_congestion_enter_rating,time_segment_2132_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-22-17-52-45.mp4,2025-10-22 17:52:45,2025-10-22 17:52:44,2025-10-22 17:53:44,2025-10-22,low,heavy delay,moderate delay,2132,train,2025-10-22 17:52:44,2025-10-22 17:53:44,2025-10-22 17:53:44,normanniles3/normanniles3_2025-10-22-17-53-45.mp4,heavy delay,0.0,60.0,2025-10-22,normanniles3,train\n0YkHaeXUMsjtnvgP8NiHuAw,Norman Niles #3,time_segment_2338_Norman Niles #3_congestion_enter_rating,time_segment_2338_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-23-09-19-45.mp4,2025-10-23 09:19:45,2025-10-23 09:19:42,2025-10-23 09:20:41,2025-10-23,none,free flowing,free flowing,2338,train,2025-10-23 09:19:42,2025-10-23 09:20:41,2025-10-23 09:20:42,normanniles3/normanniles3_2025-10-23-09-20-45.mp4,free flowing,1.0,60.0,2025-10-23,normanniles3,train\nGosHaamYHrzA698P_KuO8AM,Norman Niles #3,time_segment_2408_Norman Niles #3_congestion_enter_rating,time_segment_2408_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-23-10-32-45.mp4,2025-10-23 10:32:45,2025-10-23 10:32:42,2025-10-23 10:33:41,2025-10-23,low,free flowing,free flowing,2408,train,2025-10-23 10:32:42,2025-10-23 10:33:41,2025-10-23 10:33:41,normanniles3/normanniles3_2025-10-23-10-33-45.mp4,light delay,0.0,59.0,2025-10-23,normanniles3,train\nKYsHab-wDNymmecP6_3GoAQ,Norman Niles #3,time_segment_2657_Norman Niles #3_congestion_enter_rating,time_segment_2657_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-23-14-45-45.mp4,2025-10-23 14:45:45,2025-10-23 14:45:41,2025-10-23 14:46:40,2025-10-23,low,light delay,free flowing,2657,train,2025-10-23 14:45:41,2025-10-23 14:46:40,2025-10-23 14:46:41,normanniles3/normanniles3_2025-10-23-14-46-45.mp4,light delay,1.0,60.0,2025-10-23,normanniles3,train\n8YkHaY6KCJSgmecP8YHv4Qc,Norman Niles #3,time_segment_3340_Norman Niles #3_congestion_enter_rating,time_segment_3340_Norman Niles #3_congestion_exit_rating,normanniles3/normanniles3_2025-10-24-14-10-45.mp4,2025-10-24 14:10:45,2025-10-24 14:10:43,2025-10-24 14:11:42,2025-10-24,low,light delay,light delay,3340,train,2025-10-24 14:10:43,2025-10-24 14:11:42,2025-10-24 14:11:43,normanniles3/normanniles3_2025-10-24-14-11-45.mp4,moderate delay,1.0,60.0,2025-10-24,normanniles3,train\naYwHaYusI5__698PweWaiQ4,Norman Niles #4,time_segment_421_Norman Niles #4_congestion_enter_rating,time_segment_421_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-20-13-05-45.mp4,2025-10-20 13:05:45,2025-10-20 13:05:44,2025-10-20 13:06:43,2025-10-20,low,free flowing,free flowing,421,train,2025-10-20 13:05:44,2025-10-20 13:06:43,2025-10-20 13:06:44,normanniles4/normanniles4_2025-10-20-13-06-45.mp4,light delay,1.0,60.0,2025-10-20,normanniles4,train\n04kHaZawJtm7nvgPq8e12AU,Norman Niles #4,time_segment_608_Norman Niles #4_congestion_enter_rating,time_segment_608_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-20-16-12-45.mp4,2025-10-20 16:12:45,2025-10-20 16:12:44,2025-10-20 16:13:43,2025-10-20,none,heavy delay,free flowing,608,train,2025-10-20 16:12:44,2025-10-20 16:13:43,2025-10-20 16:13:44,normanniles4/normanniles4_2025-10-20-16-13-45.mp4,heavy delay,1.0,60.0,2025-10-20,normanniles4,train\n_osHadGTG9m7nvgPq8e12AU,Norman Niles #4,time_segment_1680_Norman Niles #4_congestion_enter_rating,time_segment_1680_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-22-10-16-45.mp4,2025-10-22 10:16:45,2025-10-22 10:16:44,2025-10-22 10:17:43,2025-10-22,low,free flowing,free flowing,1680,train,2025-10-22 10:16:44,2025-10-22 10:17:43,2025-10-22 10:17:44,normanniles4/normanniles4_2025-10-22-10-17-45.mp4,free flowing,1.0,60.0,2025-10-22,normanniles4,train\n9YwHaZy2KLOv3NoPq7G46QE,Norman Niles #4,time_segment_1947_Norman Niles #4_congestion_enter_rating,time_segment_1947_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-22-14-45-45.mp4,2025-10-22 14:45:45,2025-10-22 14:45:44,2025-10-22 14:46:43,2025-10-22,none,heavy delay,free flowing,1947,train,2025-10-22 14:45:44,2025-10-22 14:46:43,2025-10-22 14:46:44,normanniles4/normanniles4_2025-10-22-14-46-45.mp4,light delay,1.0,60.0,2025-10-22,normanniles4,train\nA40HaZ_kLbHZnvgPgt-ZqQs,Norman Niles #4,time_segment_2792_Norman Niles #4_congestion_enter_rating,time_segment_2792_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-23-17-00-45.mp4,2025-10-23 17:00:45,2025-10-23 17:00:38,2025-10-23 17:01:37,2025-10-23,low,heavy delay,heavy delay,2792,train,2025-10-23 17:00:38,2025-10-23 17:01:37,2025-10-23 17:01:38,normanniles4/normanniles4_2025-10-23-17-01-45.mp4,heavy delay,1.0,60.0,2025-10-23,normanniles4,train\n04kHaeAWn__r3w_B5ZqJDg,Norman Niles #4,time_segment_3090_Norman Niles #4_congestion_enter_rating,time_segment_3090_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-24-09-58-45.mp4,2025-10-24 09:58:45,2025-10-24 09:58:44,2025-10-24 09:59:44,2025-10-24,none,free flowing,free flowing,3090,train,2025-10-24 09:58:44,2025-10-24 09:59:44,2025-10-24 09:59:44,normanniles4/normanniles4_2025-10-24-09-59-45.mp4,free flowing,0.0,60.0,2025-10-24,normanniles4,train\n6IwHaf-fGtiD2PgPyKaayQw,Norman Niles #4,time_segment_3139_Norman Niles #4_congestion_enter_rating,time_segment_3139_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-24-10-47-45.mp4,2025-10-24 10:47:45,2025-10-24 10:47:44,2025-10-24 10:48:43,2025-10-24,low,free flowing,free flowing,3139,train,2025-10-24 10:47:44,2025-10-24 10:48:43,2025-10-24 10:48:44,normanniles4/normanniles4_2025-10-24-10-48-45.mp4,light delay,1.0,60.0,2025-10-24,normanniles4,train\nfYsHaZO6G__rld8Pu42kgAM,Norman Niles #4,time_segment_3146_Norman Niles #4_congestion_enter_rating,time_segment_3146_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-24-10-54-45.mp4,2025-10-24 10:54:45,2025-10-24 10:54:44,2025-10-24 10:55:43,2025-10-24,medium,moderate delay,free flowing,3146,train,2025-10-24 10:54:44,2025-10-24 10:55:43,2025-10-24 10:55:44,normanniles4/normanniles4_2025-10-24-10-55-45.mp4,heavy delay,1.0,60.0,2025-10-24,normanniles4,train\nSIsHaa2iJNiD2PgPyKaayQw,Norman Niles #4,time_segment_3245_Norman Niles #4_congestion_enter_rating,time_segment_3245_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-24-12-33-45.mp4,2025-10-24 12:33:45,2025-10-24 12:33:44,2025-10-24 12:34:43,2025-10-24,low,light delay,free flowing,3245,train,2025-10-24 12:33:44,2025-10-24 12:34:43,2025-10-24 12:34:44,normanniles4/normanniles4_2025-10-24-12-34-45.mp4,free flowing,1.0,60.0,2025-10-24,normanniles4,train\n24wHafcw0JPc2g_ys8igDQ,Norman Niles #4,time_segment_3281_Norman Niles #4_congestion_enter_rating,time_segment_3281_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-24-13-10-45.mp4,2025-10-24 13:10:45,2025-10-24 13:10:44,2025-10-24 13:11:43,2025-10-24,low,heavy delay,free flowing,3281,train,2025-10-24 13:10:44,2025-10-24 13:11:43,2025-10-24 13:11:44,normanniles4/normanniles4_2025-10-24-13-11-45.mp4,moderate delay,1.0,60.0,2025-10-24,normanniles4,train\n_4wHadbpCrHZnvgPgt-ZqQs,Norman Niles #4,time_segment_3303_Norman Niles #4_congestion_enter_rating,time_segment_3303_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-24-13-33-45.mp4,2025-10-24 13:33:45,2025-10-24 13:33:44,2025-10-24 13:34:44,2025-10-24,medium,heavy delay,free flowing,3303,train,2025-10-24 13:33:44,2025-10-24 13:34:44,2025-10-24 13:34:44,normanniles4/normanniles4_2025-10-24-13-34-45.mp4,moderate delay,0.0,60.0,2025-10-24,normanniles4,train\n8IkHacKcLceTmecPu8rsgQE,Norman Niles #4,time_segment_3348_Norman Niles #4_congestion_enter_rating,time_segment_3348_Norman Niles #4_congestion_exit_rating,normanniles4/normanniles4_2025-10-24-14-18-45.mp4,2025-10-24 14:18:45,2025-10-24 14:18:44,2025-10-24 14:19:43,2025-10-24,medium,moderate delay,free flowing,3348,train,2025-10-24 14:18:44,2025-10-24 14:19:43,2025-10-24 14:19:44,normanniles4/normanniles4_2025-10-24-14-19-45.mp4,moderate delay,1.0,60.0,2025-10-24,normanniles4,train\n'


In [ ]:
#-- cell 4
# --- Generate Custom ByteTrack Config ---
# This physical YAML file lets Ultralytics override the default tracker settings in Colab.
import yaml
from ultralytics import YOLO

custom_config = {
    "tracker_type": "bytetrack",
    "track_high_thresh": 0.4,
    "track_low_thresh": 0.05,   # Lowered for low-res CCTV clips.
    "new_track_thresh": 0.5,
    "track_buffer": 90,         # Keeps lost tracks alive longer through occlusion.
    "match_thresh": 0.8,
    "gmc_method": "sparseOptFlow",
    "proximity_thresh": 0.5,
    "appearance_thresh": 0.25,
    "with_reid": False,
    "fuse_score": True,
}

with open(TRACKER_CONFIG_PATH, "w") as f:
    yaml.dump(custom_config, f)
print(f"? Generated {TRACKER_CONFIG_PATH} with optimized low-res settings.")

# --- Load pre-trained YOLOv8 model ---
model = YOLO(YOLO_MODEL_NAME)
print(f"? {YOLO_MODEL_NAME} loaded successfully!")

# --- Pick a test video ---
test_video = None
for cam_name, videos in all_videos.items():
    if len(videos) > 0:
        test_video = videos[0]
        print(f"?? Test video: {make_video_key(test_video)} (from {cam_name})")
        break

if test_video is None:
    print("? No videos found! Check your Google Drive path.")

In [ ]:
# --- YOLOv8 + ByteTrack: independent clips, reproducible crossings ---
import copy
import json
import math
import re
from collections import defaultdict
from datetime import datetime, timezone

GLOBAL_TRACKING_STATE = {}
JOURNAL_RESULT_CONTEXT = None

def unsharp_mask(image, kernel_size=(5, 5), sigma=1.0, amount=1.5):
    blurred = cv2.GaussianBlur(image, kernel_size, sigma)
    return cv2.addWeighted(image, 1 + amount, blurred, -amount, 0)

def get_adaptive_params(width, height):
    pixels = width * height
    if pixels < 854 * 480:
        return {"apply_sharpen": True, "imgsz": 1280, "conf": 0.05, "line_thick": 1}
    if pixels <= 1920 * 1080:
        return {"apply_sharpen": False, "imgsz": 1280, "conf": 0.05, "line_thick": 2}
    return {"apply_sharpen": False, "imgsz": 1920, "conf": 0.05, "line_thick": 4}

def segment_intersection(a, b, c, d):
    def cross(p, q, r):
        return (q[0] - p[0]) * (r[1] - p[1]) - (q[1] - p[1]) * (r[0] - p[0])
    ab_c, ab_d = cross(a, b, c), cross(a, b, d)
    cd_a, cd_b = cross(c, d, a), cross(c, d, b)
    return ab_c * ab_d < 0 and cd_a * cd_b < 0

def line_side(a, b, point):
    return (b[0] - a[0]) * (point[1] - a[1]) - (b[1] - a[1]) * (point[0] - a[0])

def crossing_direction(a, b, old, new):
    line_x, line_y = b[0] - a[0], b[1] - a[1]
    move_x, move_y = new[0] - old[0], new[1] - old[1]
    return "exiting" if line_x * move_y - line_y * move_x > 0 else "entering"

def camera_id_from_path(video_path):
    return os.path.basename(video_path).split("_", 1)[0].lower()

def line_for_video(video_path, width, height):
    normalized = COUNTING_LINES.get(camera_id_from_path(video_path), COUNTING_LINES["default"])
    return tuple((int(round(x * width)), int(round(y * height))) for x, y in normalized)

def reset_tracker_for_new_clip(model):
    """ByteTrack's IDs/history must not carry into an unrelated video."""
    predictor = getattr(model, "predictor", None)
    trackers = getattr(predictor, "trackers", None)
    if trackers:
        for tracker in trackers:
            tracker.reset()

def generate_heatmap(points, width, height, background_frame):
    if not points or background_frame is None:
        return background_frame
    heat = np.zeros((height, width), dtype=np.float32)
    for x, y in points:
        if 0 <= x < width and 0 <= y < height:
            heat[y, x] += 1
    heat = cv2.GaussianBlur(heat, (15, 15), 0)
    if heat.max() <= 0:
        return background_frame
    colored = cv2.applyColorMap(np.uint8(255 * heat / heat.max()), cv2.COLORMAP_JET)
    result = background_frame.copy()
    blended = cv2.addWeighted(colored, 0.6, background_frame, 0.4, 0)
    result[heat > 0] = blended[heat > 0]
    return result

def process_video_with_tracking(video_path, model, max_frames=None, show_sample=False):
    """Process one full clip; return features, sample, heatmap or three Nones if interrupted."""
    global GLOBAL_TRACKING_STATE
    video_key = make_video_key(video_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        cap.release()
        raise OSError(f"Cannot open video: {video_path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if width <= 0 or height <= 0:
        cap.release()
        raise ValueError(f"Invalid video dimensions: {video_path}")
    params = get_adaptive_params(width, height)
    pt_a, pt_b = line_for_video(video_path, width, height)
    reset_tracker_for_new_clip(model)

    counts = {direction: defaultdict(int) for direction in ("entering", "exiting")}
    counted_ids, track_history, last_moved_frame = set(), {}, {}
    track_confidences = defaultdict(list)
    frame_data, all_bbox_areas, all_confidences = [], [], []
    observations, crossings, points = [], [], []
    sample_frame = background_frame = None
    frame_idx = 0
    max_frames = (total_frames if total_frames > 0 else float("inf")) if max_frames is None else max_frames
    try:
        while cap.isOpened() and frame_idx < max_frames:
            ok, frame = cap.read()
            if not ok:
                break
            if background_frame is None:
                background_frame = frame.copy()
            processed = unsharp_mask(frame) if params["apply_sharpen"] else frame
            # persist=True is valid only for successive frames *within this clip*.
            result = model.track(
                processed, persist=True, tracker=TRACKER_CONFIG_PATH,
                imgsz=params["imgsz"], conf=params["conf"], verbose=False,
                classes=list(VEHICLE_CLASSES)
            )[0]
            annotated = processed.copy() if show_sample else None
            if annotated is not None:
                cv2.line(annotated, pt_a, pt_b, (0, 255, 255), params["line_thick"])
            frame_vehicles = 0
            if result.boxes is not None and result.boxes.id is not None:
                for box in result.boxes:
                    class_id = int(box.cls[0])
                    track_id = int(box.id[0])
                    confidence = float(box.conf[0])
                    x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                    center = ((x1 + x2) // 2, (y1 + y2) // 2)
                    class_name = VEHICLE_CLASSES.get(class_id, "unknown")
                    frame_vehicles += 1
                    all_bbox_areas.append(max(0, x2 - x1) * max(0, y2 - y1))
                    all_confidences.append(confidence)
                    track_confidences[track_id].append(confidence)
                    points.append(center)
                    observations.append({
                        "frame_idx": frame_idx, "track_id": track_id,
                        "vehicle_class": class_name, "confidence": confidence,
                        "cx": center[0], "cy": center[1],
                        "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    })
                    previous = track_history.get(track_id)
                    is_stopped = False
                    if previous is None or math.hypot(center[0] - previous[0], center[1] - previous[1]) > 5:
                        last_moved_frame[track_id] = frame_idx
                    elif fps > 0 and frame_idx - last_moved_frame.get(track_id, frame_idx) > 3 * fps:
                        is_stopped = True
                    if previous is not None and track_id not in counted_ids:
                        if segment_intersection(pt_a, pt_b, previous, center):
                            direction = crossing_direction(pt_a, pt_b, previous, center)
                            counts[direction][class_name] += 1
                            counted_ids.add(track_id)
                            crossings.append({
                                "frame_idx": frame_idx, "track_id": track_id,
                                "vehicle_class": class_name, "direction": direction,
                                "crossing_confidence": confidence,
                                "previous_cx": previous[0], "previous_cy": previous[1],
                                "cx": center[0], "cy": center[1],
                            })
                    # Preserve the last off-line position when a center lands exactly
                    # on the tripwire; count once it actually reaches the other side.
                    if line_side(pt_a, pt_b, center) != 0:
                        track_history[track_id] = center
                    if annotated is not None:
                        color = (0, 0, 255) if is_stopped else (255, 0, 0)
                        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, params["line_thick"])
                        cv2.putText(annotated, f"{class_name} #{track_id}", (x1, max(15, y1 - 8)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
                        if is_stopped:
                            cv2.putText(annotated, "STOPPED VEHICLE", (x1, max(25, y1 - 25)),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            frame_data.append({"frame_idx": frame_idx, "vehicles_in_frame": frame_vehicles})
            if annotated is not None and (frame_idx == total_frames // 2 or sample_frame is None):
                sample_frame = annotated.copy()
            frame_idx += 1
    except KeyboardInterrupt:
        print(f"Interrupted at frame {frame_idx}; no partial results saved. Rerun this clip from frame zero.")
        return None, None, None
    finally:
        cap.release()

    if frame_idx == 0:
        raise ValueError(f"No frames read: {video_path}")
    for event in crossings:
        history = track_confidences[event["track_id"]]
        event["mean_track_confidence"] = float(np.mean(history))
        event["track_confidence_history"] = json.dumps(history)
    soft_entering = sum(e["mean_track_confidence"] for e in crossings if e["direction"] == "entering")
    soft_exiting = sum(e["mean_track_confidence"] for e in crossings if e["direction"] == "exiting")
    hard_entering = sum(counts["entering"].values())
    hard_exiting = sum(counts["exiting"].values())
    duration = frame_idx / fps if fps > 0 else 0.0
    densities = [row["vehicles_in_frame"] for row in frame_data]
    safe_key = re.sub(r"[^A-Za-z0-9_.-]+", "_", video_key)
    log_prefix = os.path.join(RUN_LOG_DIR, safe_key)
    features = {
        "video_path": video_key, "filename": os.path.basename(video_path),
        "camera_id": camera_id_from_path(video_path), "model_name": YOLO_MODEL_NAME,
        "total_frames": frame_idx, "fps": fps, "duration_sec": round(duration, 3),
        "resolution": f"{width}x{height}", "counting_line": f"{pt_a}->{pt_b}",
        "total_unique_vehicles": hard_entering + hard_exiting,
        "entering_count": hard_entering, "exiting_count": hard_exiting,
        "soft_count": round(soft_entering + soft_exiting, 4),
        "soft_entering_count": round(soft_entering, 4),
        "soft_exiting_count": round(soft_exiting, 4),
        "avg_vehicles_per_frame": float(np.mean(densities)),
        "max_vehicles_per_frame": int(max(densities)),
        "flow_rate_per_min": (hard_entering + hard_exiting) * 60 / duration if duration else 0,
        "avg_bbox_area": float(np.mean(all_bbox_areas)) if all_bbox_areas else 0,
        "std_bbox_area": float(np.std(all_bbox_areas)) if all_bbox_areas else 0,
        "avg_confidence": float(np.mean(all_confidences)) if all_confidences else 0,
        "crossing_log": log_prefix + "_crossings.csv",
    }
    for class_name in VEHICLE_CLASSES.values():
        features[f"{class_name}_count"] = counts["entering"][class_name] + counts["exiting"][class_name]
    manifest = {
        "video_path": video_key, "processed_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_weights": YOLO_MODEL_NAME, "tracker_config_path": TRACKER_CONFIG_PATH,
        "tracker_config": custom_config, "ultralytics_version": __import__("ultralytics").__version__,
        "counting_line_pixels": [pt_a, pt_b], "adaptive_parameters": params,
        "frames_processed": frame_idx, "fps": fps, "hard_count": hard_entering + hard_exiting,
        "soft_count_definition": "sum of mean YOLO confidences per crossed track",
        "soft_count": soft_entering + soft_exiting,
        "crossings_file": log_prefix + "_crossings.csv",
        "observations_file": log_prefix + "_observations.csv",
    }
    pd.DataFrame(crossings, columns=[
        "frame_idx", "track_id", "vehicle_class", "direction", "crossing_confidence",
        "previous_cx", "previous_cy", "cx", "cy", "mean_track_confidence",
        "track_confidence_history"
    ]).to_csv(manifest["crossings_file"], index=False)
    pd.DataFrame(observations, columns=[
        "frame_idx", "track_id", "vehicle_class", "confidence", "cx", "cy", "x1", "y1", "x2", "y2"
    ]).to_csv(manifest["observations_file"], index=False)
    with open(log_prefix + "_manifest.json", "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2)
    GLOBAL_TRACKING_STATE = {
        "video_path": video_key, "counts": counts, "frame_data": frame_data,
        "all_confidences": all_confidences, "all_bbox_areas": all_bbox_areas,
        "crossings": crossings, "features": features,
    }
    return features, sample_frame, generate_heatmap(points, width, height, background_frame)

# The single-video journal context is a snapshot, not the mutable last batch clip.
if test_video:
    features, sample, heatmap = process_video_with_tracking(test_video, model, show_sample=True)
    if features is not None:
        JOURNAL_RESULT_CONTEXT = copy.deepcopy(GLOBAL_TRACKING_STATE)
        print(f"Test clip: {features['video_path']}; hard={features['total_unique_vehicles']}; soft={features['soft_count']}")
        from google.colab.patches import cv2_imshow
        if sample is not None:
            cv2_imshow(sample)
        if heatmap is not None:
            cv2_imshow(heatmap)


In [ ]:
# --- Balanced clip extraction and genuine next-segment congestion prediction ---
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

def load_existing_features(path):
    if RESUME_FEATURE_EXTRACTION and os.path.isfile(path):
        existing = pd.read_csv(path)
        if "model_name" not in existing or not existing["model_name"].eq(YOLO_MODEL_NAME).all():
            raise ValueError("Checkpoint contains a different model; use a fresh YOLOv8 output path.")
        if "video_path" in existing.columns:
            existing = existing.drop_duplicates("video_path", keep="last")
            return existing.to_dict("records"), set(existing["video_path"].astype(str))
    return [], set()

def save_feature_checkpoint(rows, path):
    table = pd.DataFrame(rows)
    if not table.empty:
        table = table.drop_duplicates("video_path", keep="last")
        table.to_csv(path, index=False)
    return table

def prepare_next_segment_labels(labels, available_video_keys):
    """Pair each source clip with the next *contiguous* labeled segment in its camera."""
    required = {"videos", "view_label", "datetimestamp_start", "datetimestamp_end", "congestion_enter_rating"}
    missing = required - set(labels.columns)
    if missing:
        raise ValueError(f"Train.csv is missing columns: {sorted(missing)}")
    table = labels.copy()
    table["source_time"] = pd.to_datetime(table["datetimestamp_start"], errors="coerce")
    table["segment_end"] = pd.to_datetime(table["datetimestamp_end"], errors="coerce")
    table = table.dropna(subset=["source_time", "segment_end", "congestion_enter_rating"])
    table = table.sort_values(["view_label", "source_time", "videos"]).drop_duplicates("videos")
    group = table.groupby("view_label", sort=False)
    table["target_time"] = group["source_time"].shift(-1)
    table["target_video_path"] = group["videos"].shift(-1)
    table["next_enter_rating"] = group["congestion_enter_rating"].shift(-1)
    table["gap_sec"] = (table["target_time"] - table["segment_end"]).dt.total_seconds()
    table["horizon_sec"] = (table["target_time"] - table["source_time"]).dt.total_seconds()
    table = table[
        table["videos"].isin(available_video_keys)
        & table["next_enter_rating"].notna()
        & table["gap_sec"].between(0, 2)
        & (table["horizon_sec"] > 0)
    ].copy()
    table["source_date"] = table["source_time"].dt.strftime("%Y-%m-%d")
    table["camera_id"] = table["videos"].str.split("/").str[0]
    table["split"] = np.where(table["source_date"].isin(TEST_DATES), "test", "train")
    return table

def select_balanced_source_clips(pairs):
    selected = []
    for (camera_id, split, target), group in pairs.groupby(["camera_id", "split", "next_enter_rating"]):
        limit = TEST_CLIPS_PER_CAMERA_CLASS if split == "test" else TRAIN_CLIPS_PER_CAMERA_CLASS
        selected.append(group.sample(n=min(limit, len(group)), random_state=RANDOM_SEED))
    if not selected:
        return pairs.iloc[0:0].copy()
    return pd.concat(selected, ignore_index=True).sort_values(["split", "camera_id", "source_time"])

available_videos = {make_video_key(path): path for paths in all_videos.values() for path in paths}
if train_df is None:
    raise FileNotFoundError("Train.csv is needed for balanced next-segment selection")
# Exact reference selection embedded from the completed YOLO11 export.
from io import StringIO
selected_pairs = pd.read_csv(StringIO(REFERENCE_PAIRS_CSV))
if len(selected_pairs) != 80 or selected_pairs["videos"].duplicated().any():
    raise ValueError("Invalid reference selection; expected 80 unique clips.")
missing_videos = sorted(set(selected_pairs["videos"]) - set(available_videos))
if missing_videos:
    raise FileNotFoundError(
        f"{len(missing_videos)} reference clips are missing from Drive. "
        f"Restore them before comparing models. First missing: {missing_videos[:5]}"
    )
current_pairs = prepare_next_segment_labels(train_df, set(available_videos))
check_columns = ["videos", "target_video_path", "next_enter_rating",
                 "congestion_enter_rating", "split", "source_time", "target_time"]
expected = selected_pairs.set_index("videos")[check_columns[1:]].astype(str)
actual = current_pairs.set_index("videos").reindex(expected.index)[check_columns[1:]].astype(str)
if not expected.equals(actual):
    raise ValueError("Train.csv labels or split differ from the reference YOLO11 run.")
selected_pairs.to_csv(os.path.join(JOURNAL_EXPORT_DIR, "selected_next_segment_pairs.csv"), index=False)
print(f"Selected {len(selected_pairs)} source clips with a contiguous future label.")
print(selected_pairs.groupby(["split", "next_enter_rating"]).size().to_string() if len(selected_pairs) else "No eligible clips")

all_features, processed_videos = load_existing_features(FEATURES_CSV_PATH)
selected_keys = list(dict.fromkeys(selected_pairs["videos"].tolist()))
if MAX_UNLABELED_VIDEOS_PER_CAMERA > 0:
    for paths in all_videos.values():
        selected_keys.extend(make_video_key(path) for path in paths[:MAX_UNLABELED_VIDEOS_PER_CAMERA])
selected_keys = list(dict.fromkeys(selected_keys))
saved_samples = 0
for index, video_key in enumerate(selected_keys, start=1):
    if video_key in processed_videos:
        print(f"[{index}/{len(selected_keys)}] Already checkpointed: {video_key}")
        continue
    video_path = available_videos[video_key]
    print(f"[{index}/{len(selected_keys)}] Processing {video_key}")
    try:
        row, sample, _ = process_video_with_tracking(video_path, model, show_sample=saved_samples < 8)
        if row is None:
            print("Interrupted; rerun Cell 6 to restart this clip from frame zero")
            break
        all_features.append(row)
        processed_videos.add(video_key)
        save_feature_checkpoint(all_features, FEATURES_CSV_PATH)
        if sample is not None and saved_samples < 8:
            sample_path = os.path.join(ANNOTATION_DIR, f"{Path(video_path).stem}_sample.jpg")
            cv2.imwrite(sample_path, sample)
            saved_samples += 1
    except Exception as error:
        print(f"Could not process {video_key}: {type(error).__name__}: {error}")

features_df = save_feature_checkpoint(all_features, FEATURES_CSV_PATH)
print(f"Completed feature rows: {len(features_df)}; saved at {FEATURES_CSV_PATH}")
display(features_df.head())


# Never silently score a partial run or reuse another detector's checkpoint.
missing_keys = sorted(set(selected_keys) - set(features_df.get("video_path", [])))
if missing_keys:
    raise RuntimeError(
        f"{len(missing_keys)} selected clips are incomplete. Rerun Cell 6 before "
        f"exporting comparison results. First missing: {missing_keys[:5]}"
    )

FEATURE_COLUMNS = [
    "total_unique_vehicles", "entering_count", "exiting_count", "car_count",
    "truck_count", "bus_count", "motorcycle_count", "avg_vehicles_per_frame",
    "max_vehicles_per_frame", "flow_rate_per_min", "avg_bbox_area",
    "std_bbox_area", "avg_confidence", "soft_count",
]

if not features_df.empty and not selected_pairs.empty:
    labeled_features_df = features_df.merge(
        selected_pairs[["videos", "source_time", "target_time", "source_date", "split",
                        "next_enter_rating", "congestion_enter_rating", "horizon_sec"]],
        left_on="video_path", right_on="videos", how="inner", validate="one_to_one"
    )
    print(f"Matched future labels: {len(labeled_features_df)}/{len(selected_pairs)} selected clips")
    print(labeled_features_df.groupby(["split", "next_enter_rating"]).size().to_string())
    labeled_features_df.to_csv(os.path.join(JOURNAL_EXPORT_DIR, "next_segment_features.csv"), index=False)
else:
    labeled_features_df = pd.DataFrame()

if not labeled_features_df.empty:
    train_rows = labeled_features_df[labeled_features_df["split"] == "train"].copy()
    test_rows = labeled_features_df[labeled_features_df["split"] == "test"].copy()
    all_classes = sorted(train_df["congestion_enter_rating"].dropna().unique())
    training_classes = set(train_rows["next_enter_rating"])
    test_classes = set(test_rows["next_enter_rating"])
    enough = (len(train_rows) >= 20 and len(test_rows) >= 8
              and training_classes == set(all_classes) and test_classes == set(all_classes))
    if enough:
        encoder = LabelEncoder().fit(all_classes)
        X_train = train_rows[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").fillna(0)
        X_test = test_rows[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").fillna(0)
        y_train = encoder.transform(train_rows["next_enter_rating"])
        y_test = encoder.transform(test_rows["next_enter_rating"])
        xgb_model = xgb.XGBClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            objective="multi:softprob", eval_metric="mlogloss",
            tree_method="hist", random_state=RANDOM_SEED,
        )
        xgb_model.fit(X_train, y_train)
        y_pred = xgb_model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, labels=range(len(all_classes)),
                            average="macro", zero_division=0)
        persistence_pred = encoder.transform(test_rows["congestion_enter_rating"])
        persistence_accuracy = accuracy_score(y_test, persistence_pred)
        persistence_macro_f1 = f1_score(y_test, persistence_pred,
                                        labels=range(len(all_classes)), average="macro", zero_division=0)
        majority_class = train_rows["next_enter_rating"].mode().iloc[0]
        majority_pred = encoder.transform([majority_class] * len(test_rows))
        majority_accuracy = accuracy_score(y_test, majority_pred)
        print("Next-segment prediction; train dates before holdout dates:")
        print(f"Train: {len(train_rows)}; test: {len(test_rows)}; horizon range: "
              f"{test_rows['horizon_sec'].min():.0f}-{test_rows['horizon_sec'].max():.0f} sec")
        print(classification_report(y_test, y_pred, labels=range(len(all_classes)),
                                    target_names=encoder.classes_, zero_division=0))
        print("Confusion matrix (rows=true, columns=predicted):")
        print(confusion_matrix(y_test, y_pred, labels=range(len(all_classes))))
        print(f"Accuracy={accuracy:.4f}; macro-F1={macro_f1:.4f}; "
              f"Zindi-style composite={0.7 * macro_f1 + 0.3 * accuracy:.4f}")
        print(f"Persistence baseline: accuracy={persistence_accuracy:.4f}; "
              f"macro-F1={persistence_macro_f1:.4f}; "
              f"majority-class accuracy={majority_accuracy:.4f}")
        prediction_rows = test_rows[["video_path", "source_time", "target_time", "next_enter_rating"]].copy()
        prediction_rows["predicted_next_enter_rating"] = encoder.inverse_transform(y_pred)
        prediction_rows["persistence_prediction"] = test_rows["congestion_enter_rating"].values
        prediction_rows.to_csv(os.path.join(JOURNAL_EXPORT_DIR, "next_segment_holdout_predictions.csv"), index=False)
        pd.DataFrame([{
            "train_n": len(train_rows), "test_n": len(test_rows),
            "train_dates": ",".join(sorted(train_rows["source_date"].unique())),
            "test_dates": ",".join(sorted(test_rows["source_date"].unique())),
            "accuracy": accuracy, "macro_f1": macro_f1,
            "zindi_style_score": 0.7 * macro_f1 + 0.3 * accuracy,
            "persistence_accuracy": persistence_accuracy,
            "persistence_macro_f1": persistence_macro_f1,
            "majority_class_accuracy": majority_accuracy,
        }]).to_csv(os.path.join(JOURNAL_EXPORT_DIR, "next_segment_holdout_metrics.csv"), index=False)
    else:
        print("Prediction evaluation skipped: finish processing selected clips and ensure all "
              "four future classes occur in training, with at least 20 train and 8 test clips.")
        print(f"Available train/test: {len(train_rows)}/{len(test_rows)}; "
              f"train classes: {sorted(training_classes)}; test classes: {sorted(test_classes)}")


In [ ]:
#-- cell 7: Descriptive graphs and results tables (one explicitly chosen clip)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from collections import defaultdict
from IPython.display import display, HTML
import os

# ============================================================
# 0. Set up Journal Export Directory on Google Drive
# ============================================================
os.makedirs(JOURNAL_EXPORT_DIR, exist_ok=True)
print(f"📁 Exporting results for journal paper to: {JOURNAL_EXPORT_DIR}\n")

# Cell 5 freezes this context before Cell 6 processes other videos.
if JOURNAL_RESULT_CONTEXT is None:
    raise RuntimeError("Run Cell 5 to completion before exporting the journal clip")
features = JOURNAL_RESULT_CONTEXT["features"]
_counts = JOURNAL_RESULT_CONTEXT["counts"]
_frame_data = JOURNAL_RESULT_CONTEXT["frame_data"]
_all_confidences = JOURNAL_RESULT_CONTEXT["all_confidences"]
_all_bbox_areas = JOURNAL_RESULT_CONTEXT["all_bbox_areas"]
print(f"Journal clip: {features['video_path']}")

# Build per-class counts
entering_counts = dict(_counts["entering"])
exiting_counts = dict(_counts["exiting"])

all_classes = sorted(set(list(entering_counts.keys()) + list(exiting_counts.keys())))
enter_vals = [entering_counts.get(c, 0) for c in all_classes]
exit_vals = [exiting_counts.get(c, 0) for c in all_classes]
total_vals = [entering_counts.get(c, 0) + exiting_counts.get(c, 0) for c in all_classes]

# ============================================================
# STYLE CONFIG (High-Res for Publications)
# ============================================================
# Using white background for academic journal papers instead of dark theme
plt.style.use('default') 
COLORS = {
    'car': '#1f77b4',
    'motorcycle': '#ff7f0e',
    'bus': '#2ca02c',
    'truck': '#d62728',
    'unknown': '#9467bd'
}
BAR_COLORS = [COLORS.get(c, '#9467bd') for c in all_classes]

print("=" * 60)
print("  CP-1 RESULTS DASHBOARD: Vehicle Detection & Counting")
print("=" * 60)

# ============================================================
# 1. TABLE 1: Summary Results Table
# ============================================================
print("\n" + "=" * 60)
print("  TABLE 1: Video Processing Summary")
print("=" * 60)

if 'features' in globals() and features:
    summary_data = {
        "Metric": [
            "Video File",
            "Resolution",
            "Total Frames Processed",
            "FPS",
            "Duration (sec)",
            "Total Unique Vehicles Counted",
            "Soft Count (sum of crossed-track mean confidences)",
            "Avg Vehicles per Frame",
            "Avg Bounding Box Area (px)",
            "Avg Detection Confidence"
        ],
        "Value": [
            features["filename"],
            features["resolution"],
            features["total_frames"],
            features["fps"],
            features["duration_sec"],
            features["total_unique_vehicles"],
            features["soft_count"],
            features["avg_vehicles_per_frame"],
            features["avg_bbox_area"],
            f"{features['avg_confidence']:.3f}"
        ]
    }
    df_summary = pd.DataFrame(summary_data)
    # Save to CSV
    df_summary.to_csv(os.path.join(JOURNAL_EXPORT_DIR, 'table1_video_summary.csv'), index=False)
    
    display(df_summary.style.set_properties(**{
        'text-align': 'left',
        'font-size': '14px'
    }).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-size', '15px'), ('font-weight', 'bold')]}
    ]).hide(axis='index'))

# ============================================================
# 2. TABLE 2: Per-Class Vehicle Count Table
# ============================================================
print("\n" + "=" * 60)
print("  TABLE 2: Per-Class Directional Vehicle Counts")
print("=" * 60)

class_data = {
    "Vehicle Class": [c.capitalize() for c in all_classes],
    "Entering": enter_vals,
    "Exiting": exit_vals,
    "Total": total_vals
}
df_class = pd.DataFrame(class_data)

totals_row = pd.DataFrame({
    "Vehicle Class": ["TOTAL"],
    "Entering": [sum(enter_vals)],
    "Exiting": [sum(exit_vals)],
    "Total": [sum(total_vals)]
})
df_class = pd.concat([df_class, totals_row], ignore_index=True)

# Save to CSV
df_class.to_csv(os.path.join(JOURNAL_EXPORT_DIR, 'table2_class_counts.csv'), index=False)

display(df_class.style.set_properties(**{
    'text-align': 'center',
    'font-size': '14px'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center'), ('font-size', '15px'), ('font-weight', 'bold')]}
]).hide(axis='index'))

# ============================================================
# 3. FIGURE 1: Per-Class Vehicle Count (Grouped Bar Chart)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('CP-1 Results Dashboard', fontsize=20, fontweight='bold', y=0.98)

# --- Plot 1: Grouped Bar Chart ---
ax1 = axes[0, 0]
x = np.arange(len(all_classes))
width = 0.35
bars1 = ax1.bar(x - width/2, enter_vals, width, label='Entering', color='#1f77b4', edgecolor='black')
bars2 = ax1.bar(x + width/2, exit_vals, width, label='Exiting', color='#ff7f0e', edgecolor='black')
ax1.set_xlabel('Vehicle Class', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Entering vs Exiting by Vehicle Class', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([c.capitalize() for c in all_classes])
ax1.legend()
ax1.bar_label(bars1, padding=3, fontsize=10)
ax1.bar_label(bars2, padding=3, fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- Plot 2: Total Vehicle Distribution (Pie Chart) ---
ax2 = axes[0, 1]
if sum(total_vals) > 0:
    wedges, texts, autotexts = ax2.pie(
        total_vals,
        labels=[c.capitalize() for c in all_classes],
        autopct='%1.1f%%',
        colors=BAR_COLORS,
        startangle=90,
        textprops={'fontsize': 12},
        wedgeprops={'edgecolor': 'black', 'linewidth': 1.0}
    )
    for t in autotexts:
        t.set_fontsize(11)
        t.set_fontweight('bold')
    ax2.set_title('Vehicle Type Distribution', fontsize=14, fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'No data', ha='center', va='center', fontsize=14)

# --- Plot 3: Vehicles per Frame Over Time ---
ax3 = axes[1, 0]
if _frame_data:
    frame_indices = [f["frame_idx"] for f in _frame_data]
    vehicles_per_frame = [f["vehicles_in_frame"] for f in _frame_data]
    
    window = max(1, len(vehicles_per_frame) // 50)
    smoothed = np.convolve(vehicles_per_frame, np.ones(window)/window, mode='valid')
    
    ax3.fill_between(range(len(smoothed)), smoothed, alpha=0.2, color='#1f77b4')
    ax3.plot(range(len(smoothed)), smoothed, color='#1f77b4', linewidth=1.5, label=f'Smoothed (window={window})')
    ax3.set_xlabel('Frame Index', fontsize=12)
    ax3.set_ylabel('Vehicles in Frame', fontsize=12)
    ax3.set_title('Vehicle Density Over Time', fontsize=14, fontweight='bold')
    ax3.legend()
    ax3.grid(alpha=0.3)

# --- Plot 4: Confidence Score Distribution ---
ax4 = axes[1, 1]
if _all_confidences:
    ax4.hist(_all_confidences, bins=50, color='#2ca02c', edgecolor='black', linewidth=0.5, alpha=0.7)
    mean_conf = np.mean(_all_confidences)
    ax4.axvline(mean_conf, color='#d62728', linestyle='--', linewidth=2, label=f'Mean: {mean_conf:.3f}')
    ax4.set_xlabel('Confidence Score', fontsize=12)
    ax4.set_ylabel('Frequency', fontsize=12)
    ax4.set_title('Detection Confidence Distribution', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=12)
    ax4.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
# Save high-res for journal (300 DPI)
plt.savefig(os.path.join(JOURNAL_EXPORT_DIR, 'fig1_results_dashboard.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# ============================================================
# 4. FIGURE 2: Bounding Box Area Distribution
# ============================================================
fig2, (ax5, ax6) = plt.subplots(1, 2, figsize=(16, 5))

if _all_bbox_areas:
    ax5.boxplot(_all_bbox_areas, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#1f77b4', color='black'),
                whiskerprops=dict(color='black'),
                capprops=dict(color='black'),
                medianprops=dict(color='#d62728', linewidth=2),
                flierprops=dict(marker='o', markerfacecolor='#ff7f0e', markersize=3, alpha=0.5))
    ax5.set_ylabel('Bounding Box Area (pixels)', fontsize=12)
    ax5.set_title('Detection Size Distribution', fontsize=14, fontweight='bold')
    ax5.grid(axis='y', alpha=0.3)

directions = ['Entering', 'Exiting']
dir_counts = [sum(enter_vals), sum(exit_vals)]
colors_dir = ['#1f77b4', '#ff7f0e']
bars = ax6.bar(directions, dir_counts, color=colors_dir, edgecolor='black', linewidth=1.0, width=0.5)
ax6.bar_label(bars, padding=5, fontsize=12, fontweight='bold')
ax6.set_ylabel('Vehicle Count', fontsize=12)
ax6.set_title('Total Directional Flow', fontsize=14, fontweight='bold')
ax6.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(JOURNAL_EXPORT_DIR, 'fig2_bbox_and_flow.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# ============================================================
# 5. TABLE 3: Model Performance Metrics
# ============================================================
print("\n" + "=" * 60)
print("  TABLE 3: YOLOv8 Detection Confidence Statistics")
print("=" * 60)

if _all_confidences:
    conf_arr = np.array(_all_confidences)
    perf_data = {
        "Metric": [
            "Total Detections",
            "Mean Confidence",
            "Median Confidence",
            "Min Confidence",
            "Max Confidence",
            "Std Deviation",
            "Detections > 0.5 conf",
            "Detections > 0.7 conf",
            "Detections > 0.9 conf"
        ],
        "Value": [
            len(conf_arr),
            f"{np.mean(conf_arr):.4f}",
            f"{np.median(conf_arr):.4f}",
            f"{np.min(conf_arr):.4f}",
            f"{np.max(conf_arr):.4f}",
            f"{np.std(conf_arr):.4f}",
            f"{(conf_arr > 0.5).sum()} ({(conf_arr > 0.5).mean()*100:.1f}%)",
            f"{(conf_arr > 0.7).sum()} ({(conf_arr > 0.7).mean()*100:.1f}%)",
            f"{(conf_arr > 0.9).sum()} ({(conf_arr > 0.9).mean()*100:.1f}%)"
        ]
    }
    df_perf = pd.DataFrame(perf_data)
    # Save to CSV
    df_perf.to_csv(os.path.join(JOURNAL_EXPORT_DIR, 'table3_model_performance.csv'), index=False)
    
    display(df_perf.style.set_properties(**{
        'text-align': 'center',
        'font-size': '14px'
    }).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-size', '15px'), ('font-weight', 'bold')]}
    ]).hide(axis='index'))

# ============================================================
# 6. CONFIDENCE BANDS (not a confusion matrix or accuracy metric)
# ============================================================
print("\n" + "=" * 60)
print("  CONFIDENCE BANDS: Descriptive Detection Counts")
print("=" * 60)

if _all_confidences:
    conf_bins = ['Low (0.0-0.3)', 'Medium (0.3-0.6)', 'High (0.6-0.8)', 'Very High (0.8-1.0)']
    bin_edges = [0.0, 0.3, 0.6, 0.8, 1.0]
    conf_arr = np.array(_all_confidences)
    
    bin_counts = []
    for i in range(len(bin_edges) - 1):
        mask = (conf_arr >= bin_edges[i]) & (conf_arr < bin_edges[i+1])
        if i == len(bin_edges) - 2:  
            mask = (conf_arr >= bin_edges[i]) & (conf_arr <= bin_edges[i+1])
        bin_counts.append(mask.sum())
    
    cm_data = {
        "Confidence Range": conf_bins,
        "Detection Count": bin_counts,
        "Percentage": [f"{c/len(conf_arr)*100:.1f}%" if len(conf_arr) > 0 else "0%" for c in bin_counts]
    }
    df_cm = pd.DataFrame(cm_data)
    
    # Legacy filename retained for earlier extraction scripts; contents are confidence bands.
    df_cm.to_csv(os.path.join(JOURNAL_EXPORT_DIR, 'table4_confidence_matrix.csv'), index=False)
    
    def color_rows(row):
        if 'High' in row['Confidence Range'] and 'Very' not in row['Confidence Range']:
            return ['background-color: #d4edda'] * len(row)
        elif 'Very High' in row['Confidence Range']:
            return ['background-color: #c3e6cb'] * len(row)
        elif 'Medium' in row['Confidence Range']:
            return ['background-color: #ffeeba'] * len(row)
        else:
            return ['background-color: #f5c6cb'] * len(row)
    
    display(df_cm.style.apply(color_rows, axis=1).set_properties(**{
        'text-align': 'center',
        'font-size': '14px',
        'color': 'black'
    }).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-size', '15px'), ('font-weight', 'bold')]}
    ]).hide(axis='index'))
    
    fig3, ax7 = plt.subplots(figsize=(8, 6))
    matrix_data = np.array(bin_counts).reshape(-1, 1)
    
    im = ax7.imshow(matrix_data, cmap='Blues', aspect='auto')
    ax7.set_yticks(range(len(conf_bins)))
    ax7.set_yticklabels(conf_bins, fontsize=12)
    ax7.set_xticks([0])
    ax7.set_xticklabels(['Detections'], fontsize=12)
    ax7.set_title('Detection Confidence Bands', fontsize=14, fontweight='bold')
    
    for i in range(len(conf_bins)):
        red, green, blue, _ = im.cmap(im.norm(bin_counts[i]))
        luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
        text_color = 'black' if luminance > 0.5 else 'white'
        ax7.text(0, i, f"{bin_counts[i]}\n({cm_data['Percentage'][i]})",
                ha='center', va='center', fontsize=14, fontweight='bold',
                color=text_color)
    
    plt.colorbar(im, ax=ax7, label='Count')
    plt.tight_layout()
    plt.savefig(os.path.join(JOURNAL_EXPORT_DIR, 'fig3_confidence_matrix.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# ============================================================
# 7. TABLE 4: Adaptive Resolution Parameters Used
# ============================================================
print("\n" + "=" * 60)
print("  TABLE 5: Adaptive Resolution Configuration")
print("=" * 60)

res_data = {
    "Resolution Tier": ["Low (< 480p)", "HD (480p-1080p)", "4K/UHD (> 1080p)"],
    "Unsharp Mask": ["ON", "OFF", "OFF"],
    "Inference Size (imgsz)": [1280, 1280, 1920],
    "Confidence Threshold": [0.05, 0.05, 0.05],
    "Line Thickness": [1, 2, 4]
}
df_res = pd.DataFrame(res_data)
# Save to CSV
df_res.to_csv(os.path.join(JOURNAL_EXPORT_DIR, 'table5_adaptive_resolution.csv'), index=False)

display(df_res.style.set_properties(**{
    'text-align': 'center',
    'font-size': '14px'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center'), ('font-size', '15px'), ('font-weight', 'bold')]}
]).hide(axis='index'))

print("\n" + "=" * 60)
print(f"  ✅ ALL RESULTS EXPORTED TO: {JOURNAL_EXPORT_DIR}")
print("=" * 60)


### Evaluation before making accuracy claims

The detector uses pretrained COCO weights. A confidence score is **not** detection accuracy. The confidence-band table and plots above are descriptive statistics, not a confusion matrix. No mAP, MOTA, counting RMSE, weather robustness, or forecasting score should be reported unless an appropriate evaluation has actually run.

For counting, manually annotate several complete clips with `video_path`, `direction`, `vehicle_class`, and `ground_truth_count`. Compare those rows with the v2 feature CSV using `Scripts/evaluate_annotated_counts.py` from the project workspace. The script computes hard- and soft-count MAE/RMSE on the *same* clips and reports the sample size. Sample clips across cameras, lighting, and congestion classes; keep a separate holdout set.

For detection mAP, label image frames with bounding boxes in YOLO dataset format, prepare a data YAML with train/val/test splits, and run Ultralytics `model.val(data=..., split="test")` on held-out images. For MOTA, annotate per-frame track identities and use a tracking evaluator; crossing totals alone cannot produce MOTA. Weather degradation needs weather labels and a comparison of the same metric by weather group. Do not transfer metrics from another study or a pretrained model card to this dataset.

The next-segment XGBoost score in Cell 6 is produced only when balanced source clips from every class have been processed and the held-out dates contain labeled examples. It is a sampled holdout result, not a claim about all 16,076 labeled videos.


In [ ]:
# --- Complete batch summaries, readable classification matrix and portable ZIP ---
import importlib.metadata
import json
import platform
import shutil
import sys
import zipfile
import matplotlib.pyplot as plt
from datetime import datetime, timezone

expected_keys = set(selected_pairs["videos"])
if (len(expected_keys) != 80 or set(labeled_features_df["video_path"]) != expected_keys
        or "xgb_model" not in globals()):
    raise RuntimeError("Finish all 80 clips and the evaluation in Cell 6 first.")
export_path = Path(JOURNAL_EXPORT_DIR)
shutil.copy2(FEATURES_CSV_PATH, export_path / Path(FEATURES_CSV_PATH).name)
batch = labeled_features_df
camera_summary = batch.groupby("camera_id").agg(
    clips=("video_path", "size"), frames=("total_frames", "sum"),
    duration_sec=("duration_sec", "sum"),
    entering=("entering_count", "sum"), exiting=("exiting_count", "sum"),
    hard_count=("total_unique_vehicles", "sum"), soft_count=("soft_count", "sum"),
)
camera_summary.to_csv(export_path / "camera_summary.csv")
display(camera_summary)
class_counts = batch[[f"{name}_count" for name in VEHICLE_CLASSES.values()]].sum()
class_counts.rename("predicted_crossings").to_csv(export_path / "batch_vehicle_class_counts.csv")
batch.groupby(["split", "next_enter_rating"]).agg(
    clips=("video_path", "size"), hard_count=("total_unique_vehicles", "sum"),
    soft_count=("soft_count", "sum"),
).to_csv(export_path / "future_class_summary.csv")

labels = list(encoder.classes_)
matrix = confusion_matrix(y_test, y_pred, labels=range(len(labels)))
pd.DataFrame(matrix, index=labels, columns=labels).to_csv(
    export_path / "next_segment_confusion_matrix.csv", index_label="true_label"
)
pd.DataFrame(classification_report(
    y_test, y_pred, labels=range(len(labels)), target_names=labels,
    zero_division=0, output_dict=True
)).T.to_csv(export_path / "next_segment_classification_report.csv")
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(matrix, cmap="Blues")
for row in range(len(labels)):
    for col in range(len(labels)):
        r, g, b, _ = im.cmap(im.norm(matrix[row, col]))
        color = "black" if 0.2126*r + 0.7152*g + 0.0722*b > 0.5 else "white"
        ax.text(col, row, str(matrix[row, col]), ha="center", va="center",
                color=color, fontsize=16, fontweight="bold")
ax.set_xticks(range(len(labels)), labels, rotation=30, ha="right")
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel("Predicted future congestion")
ax.set_ylabel("True future congestion")
ax.set_title("YOLOv8 features + XGBoost: held-out next-segment classification")
fig.colorbar(im, ax=ax, label="Clips")
fig.tight_layout()
fig.savefig(export_path / "next_segment_confusion_matrix.png", dpi=300)
plt.show()

versions = {}
for package in ("ultralytics", "torch", "torchvision", "numpy", "pandas",
                "xgboost", "scikit-learn", "opencv-python"):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = "not installed under this package name"
manifest = {
    "model": YOLO_MODEL_NAME, "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "reference_selection_sha256": REFERENCE_SELECTION_SHA256,
    "reference_export": REFERENCE_EXPORT_NAME, "clips": len(batch),
    "train_n": len(train_rows), "test_n": len(test_rows),
    "hard_count": int(batch["total_unique_vehicles"].sum()),
    "soft_count": float(batch["soft_count"].sum()),
    "tracker": custom_config, "counting_lines": COUNTING_LINES,
    "random_seed": RANDOM_SEED, "packages": versions,
    "python": sys.version, "platform": platform.platform(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "limitations": [
        "Counts/confidences are not accuracy; no manual vehicle ground truth.",
        "Historical YOLO11 software/hardware equality is not independently verified.",
        "Large detector variants have different parameter counts.",
        "Balanced 80-clip pilot; not population-wide performance.",
    ],
}
(export_path / "run_summary.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
archive_path = Path(DRIVE_BASE) / "Journal_Results_Export_yolov8_v2.zip"
# Include per-clip settings/crossings. Large per-frame logs remain in Drive.
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(export_path.rglob("*")):
        if file.is_file():
            archive.write(file, str(Path(export_path.name) / file.relative_to(export_path)))
    for pattern in ("*_manifest.json", "*_crossings.csv"):
        for file in sorted(Path(RUN_LOG_DIR).glob(pattern)):
            archive.write(file, str(Path(export_path.name) / "crossing_logs" / file.name))
print(f"Complete: {len(batch)} clips. Share this ZIP for the YOLOv8/YOLO11 comparison:")
print(archive_path)
print("Per-frame observation logs remain in:", RUN_LOG_DIR)
# Optional browser download (the archive is already saved in Drive):
# from google.colab import files
# files.download(str(archive_path))
